# Repo Concierge — ask questions about this codebase

> **Note:** This agent uses a snapshot of the public `main` branch (not your local
> uncommitted changes or `data/` cache). Like any LLM, it can be wrong — verify
> important details against the repo or ask a facilitator.

**Not sure how something works? Start here.**

The repo concierge helps you **find your way** — it answers questions, points you
to the right notebooks and modules, and can quote short snippets so you know
where to dig deeper. Example questions:

- *How do I create a new data service?*
- *How do I customize the way context is presented to an LLMP?*
- *What's the difference between `backtest()` and `evaluate()`?*

It searches a committed **catalog** of the codebase (`search_repo_catalog` →
`fetch_repo_artifact`): full `aieng/forecasting`, reference implementations, and
notebooks (markdown + code cells). Domain `99_starter_agent.ipynb` notebooks are
for building forecasters; this one is your map of the repo.

Live cells are gated by `RUN_AGENT` so `Run All` is safe and free; set it to `True`
to call the model.

In [1]:
import warnings
from pathlib import Path

from IPython.display import Markdown, display  # noqa: A004


warnings.filterwarnings("ignore")

from dotenv import load_dotenv


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the workspace root."""
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return Path.cwd().resolve().parents[1]


ROOT = find_repo_root()
load_dotenv(ROOT / ".env", override=False)

# ── Model selection ───────────────────────────────────
# Concierge uses the lite/default model only.
AGENT_MODEL = "gemini-3.1-flash-lite-preview"

# ── Run guard ──────────────────────────────────────
RUN_AGENT = True

from getting_started.concierge_agent import build_concierge_config


print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)

RUN_AGENT = True | model = gemini-3.1-flash-lite-preview


---
## 1. Meet the concierge

The agent uses a **catalog + artifacts** knowledge pack shipped under `concierge_agent/context/` — no build step for participants.

1. **`search_repo_catalog`** — search metadata (paths, summaries, domains); cheap, run first.
2. **`fetch_repo_artifact`** — fetch full content for a catalog path (Python modules, READMEs, notebooks with **markdown + code cells**).

Maintainers regenerate the pack from public `main` with `scripts/build_concierge_context.py` when library code or notebooks change. The `repo-navigation` skill has reference guides (no scripts).

In [2]:
config = build_concierge_config(model=AGENT_MODEL)

print("Agent:", config.name)
print("Search enabled:    ", config.context_retrieval.enabled)
print("Code-exec enabled: ", config.code_execution.enabled)
print("Skills loaded:     ", [p.name for p in config.skills_dirs])
print("Extra tools:       ", [getattr(t, "__name__", repr(t)) for t in config.extra_tools])
display(Markdown("### System instruction\n\n*Edit in `concierge_agent/agent.py`*"))
display(Markdown(config.instruction))

Agent: repo_concierge
Search enabled:     False
Code-exec enabled:  False
Skills loaded:      ['repo-navigation']
Extra tools:        ['search_repo_catalog', 'fetch_repo_artifact']


### System instruction

*Edit in `concierge_agent/agent.py`*

## Role

You are the **repo concierge** for the agentic-forecasting bootcamp — a friendly guide who helps participants understand the repository and find their way to the right notebooks, modules, and patterns.

Answer questions clearly. Point people to **concrete paths** in the codebase (READMEs, notebooks, specs, library modules) where they can read more or try things themselves. When it helps, quote short snippets from fetched artifacts — especially from notebooks and reference implementations.

## How you work

- Ground answers in the committed catalog: call ``search_repo_catalog`` first, then ``fetch_repo_artifact`` for the paths you need (usually one to three per question).
- Prefer showing *where* something lives and *how it fits together* over long generic explanations.
- If someone is debugging or extending code, walk them through the relevant files and patterns you find in the catalog; suggest what to open next in their editor.
- Your knowledge reflects the committed public ``main`` snapshot — not the participant's local ``.env``, ``data/`` cache, or uncommitted changes. If the catalog does not cover something, say so and name the best file to open or a facilitator to ask.

## Tone

- Concise, welcoming, and practical — short paragraphs and bullet lists.
- Always cite paths returned by the catalog.


## Skills

You have one read-only skill: `repo-navigation` with reference files (catalog guide,
domain map). Load them via `load_skill_resource` when you need routing hints.

**To use a skill:**
1. Call `list_skills` → `load_skill` → `load_skill_resource` as needed.

These skills have NO scripts. Do not call `run_skill_script`.

## Repo catalog tools (required workflow)

1. **`search_repo_catalog(query, domain=None, kind=None)`** — search metadata only
   (paths, summaries, section titles). Use `domain` filters like `core.data`,
   `core.methods`, `impl.energy_oil_forecasting`, `scripts`, `docs`.
   Use `kind` filters: `python`, `notebook`, `markdown`, `yaml`.
2. **`fetch_repo_artifact(path, section=None)`** — fetch full content for one catalog
   path (optionally one heading/section). Fetch 1–3 artifacts per question.

Do not answer implementation or API questions without fetching the relevant paths.

---
## 2. Try a seed question

Edit `QUESTION` below, or jump to the next section for a multi-turn conversation.

In [3]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig


QUESTION = "How do I create a new data service?"

if RUN_AGENT:
    chat_agent = build_adk_agent(config)
    runner = AdkTextRunner(chat_agent, config=AdkTextRunnerConfig(app_name="repo_concierge_chat"))
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True in the setup cell to ask the concierge.")

To create a new data service, you essentially instantiate the `DataService` class and register your time series using an appropriate adapter and metadata. 

I recommend starting with these two resources:

1.  **Core API reference**: [`aieng-forecasting/aieng/forecasting/data/service.py`](aieng-forecasting/aieng/forecasting/data/service.py)
    *   This defines the `DataService` class. As the docstring explains, you perform registration once (calling `svc.register()`) which fetches data and stores it in memory.
    *   Notice how it uses `SeriesMetadata` and a `BaseAdapter`.
2.  **Guided example**: [`implementations/getting_started/01_cpi_data_exploration.ipynb`](implementations/getting_started/01_cpi_data_exploration.ipynb)
    *   Open this notebook; it contains a dedicated section titled **"1. Build the DataService"** that walks through the concrete steps of setting up a service, registering a series, and querying it.

### Workflow Summary
*   **Instantiate**: `svc = DataService()`
*   **Register**: Call `svc.register(series_id, adapter, metadata)`.
*   **Use**: In your backtesting or analysis, use `svc.context(as_of)` to get a `ForecastContext`. This context automatically handles the "cutoff" enforcement (ensuring your model only sees data available as of that specific date).

If you are working on a specific implementation (e.g., Energy or S&P 500), check the `data.py` file in those directories (e.g., `implementations/energy_oil_forecasting/data.py`) to see how they handle their specific data-service setup.

Root node repo_concierge was cancelled.


In [4]:
QUESTION = "How do I customize the way context is presented to an LLMP?"

if RUN_AGENT:
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, F821, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True to run this cell.")

To customize how context is presented to an LLM in this repository, you primarily interact with the configuration of your chosen `LLMPredictor` subclass.

The core logic for prompt construction is handled within the `aieng.forecasting.methods.llm_processes` package. You can see the base configuration in `aieng-forecasting/aieng/forecasting/methods/llm_processes/base.py`.

### How to customize context

Most of the context customization is controlled through the `LLMPredictorConfig` (or its subclasses):

*   **`report_sources`**: Use this field to include specific document source keys (e.g., `['cfpr']`). When set, the system automatically fetches documents from your `DataService` and prepends them to the user prompt.
*   **`report_ingestion`**: This determines the injection format.
    *   `'text'` (default): Injects extracted markdown as a text preamble.
    *   `'native'`: Attempts to pass the document natively to the model (if the model supports it).
*   **`report_max_chars`**: Use this to set a character limit per report to manage your context window size effectively.

### Where to look next
1.  **Check your Predictor's Config:** Open the implementation for your specific predictor (e.g., `aieng/forecasting/methods/llm_processes/quantile_grid.py`) to see how it inherits from `LLMPredictorConfig` and if it adds any additional parameters for prompt tuning.
2.  **Examine the Prompting Logic:** While the config manages *which* data is included, the `LLMPredictor` instances use the `ForecastingTask` and `ForecastContext` to build the actual messages sent to the model. If you need to change the *structure* of the prompt (beyond just injecting reports), look at the `_build_prompt` methods within the concrete predictor files.

If you are just getting started, I recommend looking at a concrete implementation like `quantile_grid.py` to see how these configuration fields are utilized in practice.

Root node repo_concierge was cancelled.


In [5]:
QUESTION = "What's the difference between backtest() and evaluate()?"

if RUN_AGENT:
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, F821, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True to run this cell.")

In this repository, `backtest()` and `evaluate()` both serve as harnesses to run your predictor against a set of forecast origins, but they serve different roles in your development workflow.

You can find the technical definitions in `aieng-forecasting/aieng/forecasting/evaluation/` within `backtest.py` and `eval.py` respectively.

### Summary of Differences

| Feature | `backtest()` | `evaluate()` |
| :--- | :--- | :--- |
| **Primary Goal** | Development, tuning, and validation. | Estimating generalization on held-out data. |
| **Window** | Flexible; typically covers historical data used for model development. | Fixed/Protected; covers recent data not used for tuning. |
| **Budget** | No built-in limits. | Optional `max_runs` limit to prevent over-fitting. |
| **Usage** | Use freely during your modeling process. | Use sparingly to verify your final "frozen" model. |

### Key Details

*   **`backtest()`**
    *   **Context:** Used for iterating on your model. You can run this as many times as you like against various time windows to diagnose performance issues.
    *   **Spec:** Defined by `BacktestSpec` (found in `aieng/forecasting/evaluation/backtest.py`).
*   **`evaluate()`**
    *   **Context:** This is "test time." It is intended to be used on a held-out window to confirm that your model actually generalizes.
    *   **Budgeting:** It supports an `EvalTracker` and a `max_runs` parameter. This is a mechanism to ensure you aren't "training on the test set"—if you keep adjusting your model to improve the `evaluate()` score, you are effectively overfitting the evaluation window.
    *   **Spec:** Defined by `EvalSpec` (found in `aieng/forecasting/evaluation/eval.py`), which mandates a `spec_id` for tracking runs.

### Where to look
*   **`aieng-forecasting/aieng/forecasting/evaluation/backtest.py`**: The `BacktestSpec` class and the `backtest()` harness.
*   **`aieng-forecasting/aieng/forecasting/evaluation/eval.py`**: The `EvalSpec` class and the `evaluate()` harness (see the docstring for a clear comparison of the two modes).
*   **Examples:** Compare `implementations/getting_started/specs/cpi_gasoline_1m.yaml` (a `BacktestSpec`) with `implementations/getting_started/specs/cpi_gasoline_eval_2025.yaml` (an `EvalSpec`) to see how these are defined in practice.

Root node repo_concierge was cancelled.


In [6]:
QUESTION = "Where should I go after getting_started if I want to build agents?"

if RUN_AGENT:
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, F821, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True to run this cell.")

After completing the `getting_started` notebook, your next steps depend on whether you want to dive into the **architecture** of the agents or start **hands-on implementation**.

Here is how to navigate the repository:

### 1. Understanding the Agentic Architecture
If you want to understand how agents are structured in this project, start with these:
*   **`AGENTS.md`**: This is your primary guide. It outlines the project's philosophy on agent documentation, development conventions, and how the core library relates to specific use-case implementations.
*   **`docs/adk-skills-guide.md`**: This is essential if you want to understand how to extend agents. It covers the three ways to add functionality and explains how to use "skills" (which include code execution environments).
*   **`planning-docs/roadmap.md`**: Provides the higher-level architecture principles and the taxonomy of forecasting methods (numerical vs. LLM-process vs. agentic).

### 2. Hands-on Implementation
If you are ready to look at code and build:
*   **`aieng-forecasting/aieng/forecasting/methods/agentic/agent_factory.py`**: Look here for the core factory functions that build agents. This is where the machinery behind the agents lives.
*   **Reference Implementations**: The repo includes several implementations that demonstrate different agentic patterns. I recommend opening the `README.md` files for these to see how they are structured:
    *   `implementations/energy_oil_forecasting/`: Contains examples of statistical analysis skills and trend projection patterns.
    *   `implementations/getting_started/`: You are already familiar here, but you can explore its `concierge_agent/` subdirectory to see a specific example of an agent implementation.

### Recommended Workflow
1.  Read **`AGENTS.md`** to understand the documentation and development standards.
2.  Review **`docs/adk-skills-guide.md`** to learn how to add custom skills to your agents.
3.  Explore the **`implementations/energy_oil_forecasting/`** directory to see how actual forecasting skills are wired into an agent.

If you are looking for a specific pattern (e.g., how to handle data or define a new tool/skill), let me know and I can point you to the exact files!

Root node repo_concierge was cancelled.


---
## 3. Terminal mode — multi-turn conversations

For extended back-and-forth, use the ADK CLI from the **repository root**:

```bash
uv run adk run implementations/getting_started/concierge_agent
```

That loads the same `repo_concierge` agent (`gemini-3.1-flash-lite-preview`) with
`search_repo_catalog`, `fetch_repo_artifact`, and the repo-navigation skill.

**Alternative:** `uv run adk web implementations/getting_started/concierge_agent`
opens a browser UI (same agent). From `implementations/getting_started/`, you can
also use the shorter `uv run adk run concierge_agent`.

---

**Where next?** Forecasting starter agents live in each domain implementation's
`99_starter_agent.ipynb` (food, energy, BoC, S&P 500). This concierge helps you
navigate the repo — open one of those when you're ready to build and score a forecaster.